## Import de librerías y carga del dataset

In [75]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE
import scipy.stats as stats
import time

In [76]:
# CARGA DEL DATASET
df = pd.read_csv('/content/Dataset_ALDIMI_Merged_Clean.csv')

df.head()

,Patient_ID,Cancer_Type,Age,Gender,Smoking,Alcohol_Use,Obesity,Family_History,Diet_Red_Meat,Diet_Salted_Processed,...,BRCA_Mutation,H_Pylori_Infection,Calcium_Intake,Overall_Risk_Score,BMI,Physical_Activity_Level,Risk_Level,county_STATE,county_CTYNAME,county_POPESTIMATE2015
0,1001,Lung,52,0,10,7,4,0,10,3,...,0,0,0,0.601460,27.2,0,Medium,19,Johnson County,144251.0
1,1002,Colon,73,1,5,4,8,0,7,4,...,0,0,3,0.380986,28.7,8,Medium,51,Richmond city,220289.0
2,1003,Skin,70,0,3,0,2,0,4,1,...,0,0,7,0.265377,26.8,1,Low,40,Oklahoma County,776864.0
3,1004,Skin,64,1,6,3,4,0,4,3,...,0,0,6,0.397763,26.3,0,Medium,36,New York,19795791.0
4,1005,Skin,60,1,4,8,4,1,1,4,...,0,0,7,0.484590,27.3,2,Medium,6,San Diego County,3299521.0


In [77]:
# INFORMACIÓN DEL DATASET

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Patient_ID               3000 non-null   int64  
 1   Cancer_Type              3000 non-null   object 
 2   Age                      3000 non-null   int64  
 3   Gender                   3000 non-null   int64  
 4   Smoking                  3000 non-null   int64  
 5   Alcohol_Use              3000 non-null   int64  
 6   Obesity                  3000 non-null   int64  
 7   Family_History           3000 non-null   int64  
 8   Diet_Red_Meat            3000 non-null   int64  
 9   Diet_Salted_Processed    3000 non-null   int64  
 10  Fruit_Veg_Intake         3000 non-null   int64  
 11  Physical_Activity        3000 non-null   int64  
 12  Air_Pollution            3000 non-null   int64  
 13  Occupational_Hazards     3000 non-null   int64  
 14  BRCA_Mutation           

In [78]:
# VARIABLES CATEGÓRICAS
df.select_dtypes(include='object').columns

Index(['Cancer_Type', 'Risk_Level', 'county_CTYNAME'], dtype='object')

In [79]:
df = df.drop(columns=["Patient_ID", "county_CTYNAME"])

## FEATURE ENGINEERING

Normalización/Estandarización (Z-score).

In [80]:
X = df.drop(columns=["Risk_Level"])
y = df["Risk_Level"]

In [81]:
scaler = StandardScaler()

num_cols = X.select_dtypes(include=["int64", "float64"]).columns

X[num_cols] = scaler.fit_transform(X[num_cols])

In [82]:
# media
X[num_cols].mean()

,0
Age,-2.966516e-16
Gender,-1.361874e-16
Smoking,6.750156e-17
Alcohol_Use,7.697546e-18
Obesity,4.855375e-17
Family_History,3.079019e-17
Diet_Red_Meat,-8.171241e-17
Diet_Salted_Processed,2.486900e-17
Fruit_Veg_Intake,-8.526513e-17
Physical_Activity,1.657933e-17


In [83]:
# desviación
X[num_cols].std()

,0
Age,1.000167
Gender,1.000167
Smoking,1.000167
Alcohol_Use,1.000167
Obesity,1.000167
Family_History,1.000167
Diet_Red_Meat,1.000167
Diet_Salted_Processed,1.000167
Fruit_Veg_Intake,1.000167
Physical_Activity,1.000167


Codificación de variables (One-Hot Encoding / Label Encoding).

In [84]:
# Label Encoding para variables objetivo y el modelo pueda identificarlas
X = df.drop(columns=["Risk_Level"])
y = df["Risk_Level"]

le = LabelEncoder()
y = le.fit_transform(y)

# one hot encoding en variables categóricas para evitar generar una gerarquía falsa
X = pd.get_dummies(X, columns=["Cancer_Type"])


Manejo de desbalanceo de clases mediante SMOTE o sub-muestreo.

In [85]:
#dividimos el data set de entrenamiento para saber la cantidad de entradas de cada tipo
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [86]:
print("Cantidad de tipos de entrada")
unique, counts = np.unique(y_train, return_counts=True)

class_dist = pd.DataFrame({
    "Clase": unique,
    "Cantidad": counts
})

class_dist

Cantidad de tipos de entrada


,Clase,Cantidad
0,0,118
1,1,387
2,2,1895


La cantidad de cada tipo de entradas está muy desbalanceada por lo que se usará SMOTE para balancearlos

In [87]:
smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(X_train, y_train)

In [88]:
print("Después de SMOTE")
unique, counts = np.unique(y_train, return_counts=True)
class_dist = pd.DataFrame({
    "Clase": unique,
    "Cantidad": counts
})

class_dist

Después de SMOTE


,Clase,Cantidad
0,0,1895
1,1,1895
2,2,1895


## Modelado: Entrenar y comparar al menos dos algoritmos (Random Forest vs. XGBoost).

In [89]:
#Entrenamiento de Random forest
print("Entrenando Random Forest")
start = time.time()

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)

end = time.time()
y_pred_rf = rf.predict(X_test)

print("Random Forest listo")
print("Tiempo de entrenamiento:", round(end - start, 4), "segundos")

Entrenando Random Forest
Random Forest listo
Tiempo de entrenamiento: 1.8022 segundos


In [90]:
#Entrenamiento de XGBoost
print("Entrenando XGBoost")
start = time.time()
xgb = XGBClassifier(
    random_state=42,
    eval_metric='logloss'
)
xgb.fit(X_train, y_train)
end = time.time()
y_pred_xgb = xgb.predict(X_test)

print("XGBoost listo")
print("Tiempo de entrenamiento:", round(end - start, 4), "segundos")

Entrenando XGBoost
XGBoost listo
Tiempo de entrenamiento: 0.9052 segundos


Comparación de resultados

In [97]:
print("RANDOM FOREST")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1-score:", f1_score(y_test, y_pred_rf, average="weighted"))
print(classification_report(y_test, y_pred_rf))

print("XGBOOST ")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("F1-score:", f1_score(y_test, y_pred_xgb, average="weighted"))
print(classification_report(y_test, y_pred_xgb))

RANDOM FOREST
Accuracy: 1.0
F1-score: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        30
           1       1.00      1.00      1.00        97
           2       1.00      1.00      1.00       473

    accuracy                           1.00       600
   macro avg       1.00      1.00      1.00       600
weighted avg       1.00      1.00      1.00       600

XGBOOST 
Accuracy: 0.9983333333333333
F1-score: 0.9983200890113888
              precision    recall  f1-score   support

           0       1.00      0.97      0.98        30
           1       1.00      1.00      1.00        97
           2       1.00      1.00      1.00       473

    accuracy                           1.00       600
   macro avg       1.00      0.99      0.99       600
weighted avg       1.00      1.00      1.00       600



In [101]:
import pandas as pd

results = pd.DataFrame({
    "Modelo": ["Random Forest", "XGBoost"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_xgb)
    ],
    "F1-score": [
        f1_score(y_test, y_pred_rf, average="weighted"),
        f1_score(y_test, y_pred_xgb, average="weighted")
    ]
})

results

,Modelo,Accuracy,F1-score
0,Random Forest,1.000000,1.00000
1,XGBoost,0.998333,0.99832


El modelo Random Forest alcanzó un desempeño perfecto en el conjunto de prueba, lo que sugiere una alta capacidad de separación entre clases en el dataset. Sin embargo, este resultado también puede indicar la presencia de variables altamente predictivas o posibles correlaciones fuertes entre atributos y la variable objetivo, por lo que se recomienda validar el modelo con técnicas adicionales como cross-validation.